#  Big Data con PySpark — Notebook 1
## Carga y Exploración del Dataset de Vuelos Colombia

**Dataset:** 500,000 registros de vuelos domésticos en Colombia (2022–2023)  
**Variables:** aerolinea, origen, destino, pasajeros, retraso, tarifa, clase

---

## Paso 0 — Generar el dataset

Ejecuta esta celda **una sola vez** para crear el archivo CSV que usaremos en todos los notebooks.

In [12]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)
n = 500_000

aeropuertos = ['BOG','MDE','CLO','CTG','BAQ','SMR','PEI','VVC','LET','MTR']
aerolineas  = ['Avianca','LATAM','Wingo','EasyFly','Satena','JetBlue']
estados     = ['A_TIEMPO','DEMORADO','CANCELADO','DESVIADO']
fechas_base = pd.date_range('2022-01-01', '2023-12-31', freq='h')

df = pd.DataFrame({
    'vuelo_id':     range(1, n+1),
    'fecha':        np.random.choice(fechas_base.astype(str), n),
    'aerolinea':    np.random.choice(aerolineas,  n, p=[0.35,0.25,0.15,0.10,0.10,0.05]),
    'origen':       np.random.choice(aeropuertos, n),
    'destino':      np.random.choice(aeropuertos, n),
    'pasajeros':    np.random.randint(50, 180, n),
    'distancia_km': np.random.randint(80, 2500, n),
    'retraso_min':  np.where(np.random.random(n) < 0.3, np.random.randint(1,240,n), 0),
    'estado':       np.random.choice(estados, n, p=[0.68,0.22,0.07,0.03]),
    'tarifa_usd':   np.round(np.random.uniform(50, 800, n), 2),
    'clase':        np.random.choice(['Economica','Business','Primera'], n, p=[0.75,0.20,0.05]),
})

# Introducir nulos realistas
df.loc[np.random.choice(n, 5000, replace=False), 'retraso_min'] = np.nan
df.loc[np.random.choice(n, 3000, replace=False), 'tarifa_usd']  = np.nan
df.loc[np.random.choice(n, 1000, replace=False), 'clase']       = None

ruta = "/tmp/vuelos_colombia.csv"
df.to_csv(ruta, index=False)
print(f" Dataset creado: {len(df):,} filas  |  {os.path.getsize(ruta)/1e6:.1f} MB en disco")

 Dataset creado: 500,000 filas  |  40.0 MB en disco


---
## Paso 1 — Iniciar SparkSession

In [13]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Vuelos_Colombia")           # nombre que aparece en Spark UI (localhost:4040)
    .master("local[*]")                   # usa todos los cores de la máquina
    .config("spark.sql.shuffle.partitions", "8")  # reduce las 200 particiones por default
    .getOrCreate()                        # crea o reutiliza sesión existente
)

print("Spark versión:", spark.version)
print("Cores disponibles:", spark.sparkContext.defaultParallelism)

Spark versión: 4.0.4
Cores disponibles: 2


**Nota sobre `shuffle.partitions = 8`:**  
Por defecto Spark usa 200 particiones cuando hace operaciones como `groupBy` o `join`. Para datasets < 1 GB en modo local, eso genera mucho overhead. Usamos 8 (uno por core típico) para que sea rápido en local.

---
## Paso 2 — Leer el CSV con PySpark

In [14]:
df = (
    spark.read                    # punto de entrada para leer datos
    .option("header", "true")     # la primera fila es el encabezado
    .option("inferSchema", "true")# Spark lee el CSV dos veces: 1ra para inferir tipos
    .option("nullValue", "")      # trata strings vacíos como null
    .csv("/tmp/vuelos_colombia.csv")  # ruta del archivo
)

# df es un DataFrame de Spark — NO está en memoria todavía (lazy evaluation)
# Solo se leerá cuando llamemos una acción (.show, .count, etc.)
print(type(df))

<class 'pyspark.sql.classic.dataframe.DataFrame'>


### Opciones importantes de `.read.csv()`

| Opción | Valor | Qué hace |
|---|---|---|
| `header` | `true/false` | Si la primera fila es encabezado |
| `inferSchema` | `true/false` | Infiere tipos automáticamente (lento pero conveniente) |
| `sep` | `;` `,` `\t` | Separador del CSV |
| `nullValue` | `"NA"` `""` | Qué valor tratar como nulo |
| `encoding` | `UTF-8` | Codificación del archivo |
| `dateFormat` | `yyyy-MM-dd` | Formato de fechas |

---
## Paso 3 — Exploración inicial

In [15]:
# ── Ver el esquema (tipos de datos) ──
df.printSchema()
# printSchema() → imprime el árbol de columnas con sus tipos
# StringType, IntegerType, DoubleType, TimestampType, etc.
# ← Esta es una acción: SÍ ejecuta Spark

root
 |-- vuelo_id: integer (nullable = true)
 |-- fecha: timestamp (nullable = true)
 |-- aerolinea: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- destino: string (nullable = true)
 |-- pasajeros: integer (nullable = true)
 |-- distancia_km: integer (nullable = true)
 |-- retraso_min: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- tarifa_usd: double (nullable = true)
 |-- clase: string (nullable = true)



In [16]:
# ── Dimensiones del dataset ──
filas = df.count()              # cuenta todas las filas → acción
columnas = len(df.columns)      # df.columns → lista Python con nombres de columnas

print(f"Filas:    {filas:,}")
print(f"Columnas: {columnas}")
print(f"Columnas: {df.columns}")

Filas:    500,000
Columnas: 11
Columnas: ['vuelo_id', 'fecha', 'aerolinea', 'origen', 'destino', 'pasajeros', 'distancia_km', 'retraso_min', 'estado', 'tarifa_usd', 'clase']


In [17]:
# ── Ver las primeras filas ──
df.show(5, truncate=False)
# show(n)              → muestra n filas en formato tabla
# truncate=False       → no corta el texto de columnas largas
# truncate=30          → corta a 30 caracteres (útil con texto largo)

+--------+-------------------+---------+------+-------+---------+------------+-----------+--------+----------+---------+
|vuelo_id|fecha              |aerolinea|origen|destino|pasajeros|distancia_km|retraso_min|estado  |tarifa_usd|clase    |
+--------+-------------------+---------+------+-------+---------+------------+-----------+--------+----------+---------+
|1       |2023-10-21 03:00:00|LATAM    |LET   |PEI    |144      |1977        |0.0        |A_TIEMPO|402.64    |Economica|
|2       |2022-02-05 20:00:00|LATAM    |CTG   |PEI    |135      |611         |0.0        |A_TIEMPO|112.25    |Business |
|3       |2022-08-13 14:00:00|Wingo    |MDE   |VVC    |117      |1082        |0.0        |A_TIEMPO|640.61    |Economica|
|4       |2023-05-14 12:00:00|EasyFly  |BAQ   |VVC    |128      |1952        |235.0      |DEMORADO|506.95    |Economica|
|5       |2023-04-16 04:00:00|Avianca  |MTR   |BOG    |150      |1756        |2.0        |A_TIEMPO|183.47    |Economica|
+--------+-------------------+--

In [18]:
# ── Estadísticas descriptivas ──
df.describe(
    "pasajeros", "distancia_km", "retraso_min", "tarifa_usd"
).show()
# describe(cols...) → calcula count, mean, stddev, min, max
# Solo en columnas numéricas y string
# Si no se pasan columnas → calcula en TODAS

+-------+-----------------+----------------+------------------+-----------------+
|summary|        pasajeros|    distancia_km|       retraso_min|       tarifa_usd|
+-------+-----------------+----------------+------------------+-----------------+
|  count|           500000|          500000|            495000|           497000|
|   mean|       114.556446|     1289.711518|36.027278787878785|424.7656981690135|
| stddev|37.52524411828643|698.789677426745| 66.65315849962349|216.3863490966688|
|    min|               50|              80|               0.0|             50.0|
|    max|              179|            2499|             239.0|            800.0|
+-------+-----------------+----------------+------------------+-----------------+



In [19]:
# ── Contar nulos por columna ──
from pyspark.sql import functions as F
# F es la convención estándar para importar pyspark.sql.functions
# Contiene: col(), lit(), when(), count(), sum(), avg(), etc.

nulos = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
# Para cada columna c:
#   F.col(c)         → referencia a la columna
#   .isNull()        → True si el valor es nulo
#   F.when(cond, v)  → si cond es True, devuelve v (aquí devuelve el nombre c)
#   F.count(...)     → cuenta los no-nulos del when → cuenta cuántos nulos hay
#   .alias(c)        → renombra la columna de salida
# select([lista])    → selecciona varias expresiones a la vez

nulos.show()

+--------+-----+---------+------+-------+---------+------------+-----------+------+----------+-----+
|vuelo_id|fecha|aerolinea|origen|destino|pasajeros|distancia_km|retraso_min|estado|tarifa_usd|clase|
+--------+-----+---------+------+-------+---------+------------+-----------+------+----------+-----+
|       0|    0|        0|     0|      0|        0|           0|       5000|     0|      3000| 1000|
+--------+-----+---------+------+-------+---------+------------+-----------+------+----------+-----+



In [20]:
# ── Valores únicos en columnas categóricas ──
for col in ["aerolinea", "origen", "estado", "clase"]:
    n_unicos = df.select(col).distinct().count()
    # .distinct() → elimina duplicados → acción necesaria: .count()
    print(f"{col:15s}: {n_unicos} valores únicos")

aerolinea      : 6 valores únicos
origen         : 10 valores únicos
estado         : 4 valores únicos
clase          : 4 valores únicos


In [21]:
# ── Distribución de una categórica ──
(
    df.groupBy("aerolinea")          # agrupa por aerolinea
    .count()                          # cuenta filas por grupo
    .orderBy(F.desc("count"))         # ordena de mayor a menor
    .show()
)
# groupBy + count() es el equivalente al value_counts() de pandas
# F.desc("count") → función de orden descendente sobre la columna "count"

+---------+------+
|aerolinea| count|
+---------+------+
|  Avianca|174785|
|    LATAM|125055|
|    Wingo| 75006|
|  EasyFly| 50044|
|   Satena| 49826|
|  JetBlue| 25284|
+---------+------+



---
## Paso 4 — Cachear el DataFrame

Cada vez que llamamos una acción, Spark re-lee el CSV desde disco. Para evitar eso en análisis exploratorio, cacheamos el DataFrame en memoria.

In [22]:
df.cache()
# .cache() → marca el DataFrame para ser guardado en memoria RAM de los executors
# Equivalente a .persist(StorageLevel.MEMORY_AND_DISK)
# La primera acción después de .cache() lo materializa; las siguientes son rápidas

# Forzamos la materialización con una acción
df.count()
print("DataFrame cacheado ")

# Para liberar la caché cuando ya no la necesites:
# df.unpersist()

DataFrame cacheado 


### ¿Cuándo usar `.cache()`?

| Situación | ¿Cachear? |
|---|---|
| Vas a usar el mismo DF en múltiples operaciones |  Sí |
| El DF es el resultado de una transformación costosa |  Sí |
| Solo lo usarás una vez |  No |
| El DF no cabe en RAM |  No (usar `.persist(DISK_ONLY)`) |

---
## Resumen del notebook

```
spark.read.option(...).csv(ruta)  →  cargar CSV
df.printSchema()                  →  ver tipos de datos
df.count()                        →  contar filas
df.show(n)                        →  ver primeras filas
df.describe(cols)                 →  estadísticas descriptivas
df.groupBy(col).count()           →  distribución de categóricas
df.cache()                        →  guardar en memoria para reusar
```

 **Siguiente:** Selección, filtros y limpieza de datos